# 11 多模态图片识别

**用途：** 离线验证图片格式安全、质量信号、结构化Schema和拒识规则，不消耗视觉API。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from image_evidence import validate_image_upload

fixture_dir = PROJECT2_ROOT / "tests" / "fixtures" / "multimodal"
image_rows = []
for path in sorted(fixture_dir.glob("*.png")):
    validated = validate_image_upload(
        path.read_bytes(),
        filename=path.name,
        claimed_mime_type="image/png",
    )
    metadata = validated.public_metadata()
    image_rows.append({
        "文件": path.name,
        "尺寸": f"{metadata['width']}x{metadata['height']}",
        "字节": metadata["size_bytes"],
        "本地质量": metadata["local_quality"],
        "质量信号": "、".join(metadata["quality_signals"]),
    })
show_table(image_rows)
check_equal("四张合成图片通过安全解码", len(image_rows), 4)

,文件,尺寸,字节,本地质量,质量信号
0,synthetic_blurry_nameplate.png,1086x1448,1832920,fair,possibly_blurry
1,synthetic_damage.png,1536x1024,2484749,good,
2,synthetic_nameplate.png,1536x1024,3204535,good,
3,synthetic_part_label.png,1536x1024,2680931,good,


[PASS] 四张合成图片通过安全解码 | actual=4, expected=4


{'检查项': '四张合成图片通过安全解码', '状态': 'PASS', '说明': 'actual=4, expected=4'}

In [3]:
from pydantic import ValidationError
from schemas import ImageInspectionResult
from multimodal_evaluation import predict_rejection

evidence = ImageInspectionResult(
    image_type="nameplate",
    extracted_text=["KOMATSU", "PC200"],
    brand="KOMATSU",
    machine_model="PC200",
    part_name_candidate="液压泵",
    part_number="708-2L-00300",
    image_quality="good",
    confidence=0.92,
    safe_for_auto_merge=True,
)
print(evidence.model_dump_json(indent=2))
check("清晰铭牌不拒识", not predict_rejection(evidence.model_dump()))

invalid_blocked = False
try:
    ImageInspectionResult(
        image_type="nameplate",
        image_quality="good",
        confidence=1.5,
        safe_for_auto_merge=True,
    )
except ValidationError as exc:
    invalid_blocked = True
    print(exc)
check("越界置信度被Pydantic拒绝", invalid_blocked)

{
  "image_type": "nameplate",
  "extracted_text": [
    "KOMATSU",
    "PC200"
  ],
  "brand": "KOMATSU",
  "machine_model": "PC200",
  "part_name_candidate": "液压泵",
  "part_number": "708-2L-00300",
  "visible_damage": [],
  "observed_features": [],
  "image_quality": "good",
  "confidence": 0.92,
  "warnings": [],
  "required_followups": [],
  "safe_for_auto_merge": true
}
[PASS] 清晰铭牌不拒识
1 validation error for ImageInspectionResult
confidence
  Input should be less than or equal to 1 [type=less_than_equal, input_value=1.5, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal
[PASS] 越界置信度被Pydantic拒绝


{'检查项': '越界置信度被Pydantic拒绝', '状态': 'PASS', '说明': ''}

In [4]:
multimodal_tests = run_unittest(
    ["tests.test_multimodal_runtime"],
    project2_root=PROJECT2_ROOT,
)
check("多模态运行时10条通过", "Ran 10 tests" in multimodal_tests.output and "OK" in multimodal_tests.output)

$ D:\new things\项目1\day1\.venv\Scripts\python.exe -m unittest tests.test_multimodal_runtime -v
test_customer_isolation_and_deletion (tests.test_multimodal_runtime.ImageEvidenceRepositoryTests.test_customer_isolation_and_deletion) ... ok
test_schema_forbids_unknown_fields_and_invalid_confidence (tests.test_multimodal_runtime.ImageSchemaTests.test_schema_forbids_unknown_fields_and_invalid_confidence) ... ok
test_accepts_and_reencodes_supported_png (tests.test_multimodal_runtime.ImageValidationTests.test_accepts_and_reencodes_supported_png) ... ok
test_rejects_corrupt_or_tiny_images (tests.test_multimodal_runtime.ImageValidationTests.test_rejects_corrupt_or_tiny_images) ... ok
test_rejects_extension_mime_and_content_mismatch (tests.test_multimodal_runtime.ImageValidationTests.test_rejects_extension_mime_and_content_mismatch) ... ok
test_rejects_files_over_byte_budget (tests.test_multimodal_runtime.ImageValidationTests.test_rejects_files_over_byte_budget) ... ok
test_confirmed_image_fields

{'检查项': '多模态运行时10条通过', '状态': 'PASS', '说明': ''}

## 真实链路

JPG/PNG/WebP先校验扩展名、MIME、真实格式、字节、尺寸、动画和质量，再清理EXIF并重编码。视觉模型只提取候选证据；清晰候选仍要客户确认，模糊图要求重拍或转人工。图片二进制不写checkpoint。

`RUN_LIVE_MODEL_TESTS=False`时本Notebook不会调用智谱。真实API只证明接口和Schema跑通，不代表字段准确率。

### 面试官会问

1. 为什么不全量把DeepSeek替换为多模态模型？
2. `ImageInspectionResult`为什么禁止未知字段？
3. 零件号、铭牌、旧件标签和损坏证据如何区分？
4. 模糊、反光和遮挡时如何拒识？
5. 为什么视觉结果不能直接触发报价或适配结论？

### 参考答案

1. **为什么不全量替换DeepSeek？** 文本意图解析和回复已经由DeepSeek稳定承担，视觉只在有图片时触发。ModelRouter按`text/vision`能力分流，可以分别控制模型、费用、超时和降级，避免每个文本请求都支付多模态成本，也降低一次换模的回归风险。
2. **为什么Schema禁止未知字段？** `extra="forbid"`让Provider返回的新字段、拼写错误或Prompt漂移立即暴露，而不是悄悄混入State。置信度范围、枚举和列表结构也由Pydantic验证，失败进入受控重试或人工兜底。
3. **四类证据如何区分？** 铭牌描述整机品牌、型号、序列等；旧件标签是零件包装或旧件上的标识；零件号是必须尽量逐字符保留的精确字段；损坏证据只描述图片可见的裂纹、锈蚀、泄漏等现象，不能推断根因和责任。
4. **模糊、反光和遮挡如何拒识？** 上传阶段先检查尺寸、像素和文件合法性，模型再输出`image_quality/confidence`和可读字段；低于门槛、关键字段不可读或证据冲突时不自动合并槽位，回复补拍建议，客户可重拍或转人工。
5. **为什么不能直接报价或判适配？** 视觉OCR可能错一位件号，错误适配的业务风险高。模型结果只是候选证据，必须经过质量门控和客户确认；报价仍调用报价工具，适配仍需要RAG/规则和必要时人工核对。

**代码落点：** `model_router.py`、`schemas.py::ImageInspectionResult`、`image_evidence.py`、`vision_service.py`和`agent_graph.py`的图片节点。